# 칼로리 소모량 예측 AI — 생리학 공식 역공학 & 잔차 학습

**AI 헬스케어 머신러닝 해커톤 | 3위 입상 | 검증 RMSE 0.0529**

---

## 흐름

처음에는 XGBoost, LightGBM 같은 트리 계열 모델로 시작했다.  
아무리 튜닝해도 RMSE 1.0 아래로 내려가지 않았고, 방향을 완전히 바꿨다.

결국 데이터가 특정 생리학 공식(Keytel 공식)으로 만들어졌다는 것을 논문을 통해 찾아냈고,  
그 공식을 베이스 모델로 삼아 잔차만 머신러닝으로 보정하는 하이브리드 구조로 RMSE 0.0529를 달성했다.

```
1차 시도 : Polynomial Features + Ridge/Linear 앙상블  → RMSE 0.18
2차 시도 : 피처 축소 + 앙상블 비율 조정              → RMSE 0.09
3차 시도 : Keytel 공식 역공학 + CatBoost 잔차 학습   → RMSE 0.0529 ✅ 최종 제출
미제출   : ElasticNet 기반 잔차 학습 추가 실험
```

---

**내가 했던 파트**  
- 체온·심박수 등 생리학적 변수의 의미 해석 및 근거 리서치  
- 전처리 전략 수립 및 피처 엔지니어링 설계  
- Keytel 공식 논문 탐색 및 데이터 역공학 적용  

> 📄 Keytel LR et al. (2005). *Prediction of energy expenditure from heart rate monitoring during submaximal exercise.* Journal of Sports Sciences, 23(3), 289–297.

---
## 1차 시도 — Polynomial Features + 앙상블

### 접근
변수 간 상호작용을 3차 Polynomial Feature로 확장하고,  
LinearRegression + Ridge를 6:4로 앙상블한 후 반올림 적용.

### 결과 및 한계
RMSE 0.18까지는 내려왔지만 그 이상 개선이 안 됐다.  
나중에 알고 보니 파생 변수를 많이 만들수록 좋아진다는 가정 자체가 잘못됐다.  
4개 기본 변수(체중, 나이, BPM, 운동시간)의 재조합이라 다중공선성 문제가 심각했다.

In [ ]:
import os, random, warnings
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)

seed_everything(42)

In [ ]:
DATA_DIR = "/content/"
train = pd.read_csv(DATA_DIR + 'train.csv')
test  = pd.read_csv(DATA_DIR + 'test.csv')

train_x = train.drop(columns=["ID", "Calories_Burned"])
train_y = train["Calories_Burned"]
test_x  = test.drop(columns=["ID"])

In [ ]:
# Weight_Status: 순서가 있는 범주형이라 Ordinal Encoding 적용
status_map = {"Normal Weight": 0, "Overweight": 1, "Obese": 2}
train_x["Weight_Status"] = train_x["Weight_Status"].map(status_map)
test_x["Weight_Status"]  = test_x["Weight_Status"].map(status_map)

le = LabelEncoder()
train_x["Gender"] = le.fit_transform(train_x["Gender"])
test_x["Gender"]  = le.transform(test_x["Gender"])

X_train, X_val, y_train, y_val = train_test_split(
    train_x, train_y, test_size=0.2, random_state=42
)

# 3차 다항 피처 생성 — 변수 간 상호작용 추가
poly = PolynomialFeatures(degree=3, interaction_only=True, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_val_poly   = poly.transform(X_val)

In [ ]:
lr    = LinearRegression()
ridge = Ridge(alpha=0.1)

lr.fit(X_train_poly, y_train)
ridge.fit(X_train_poly, y_train)

val_preds       = 0.6 * lr.predict(X_val_poly) + 0.4 * ridge.predict(X_val_poly)
final_val_preds = np.round(val_preds)

rmse = np.sqrt(mean_squared_error(y_val, final_val_preds))
print(f"1차 시도 Validation RMSE: {rmse:.5f}")

---
## 2차 시도 — 피처 축소 + 앙상블 비율 조정

### 접근
1차에서 다중공선성 문제를 확인한 후, 체온·키 관련 변수를 제거하고  
핵심 변수(BPM, 운동시간, 체중, 나이, 성별)만 남겼다.  
앙상블 비율도 조정하고 Ridge alpha도 튜닝했다.

### 결과 및 한계
RMSE 0.09까지 개선됐지만 여전히 벽이 있었다.  
이 시점에서 '더 복잡한 모델'이 아니라 '데이터 자체의 구조'를 의심하기 시작했다.

In [ ]:
# 노이즈로 판단된 변수 제거
# 체온(F), 키 관련 변수, 체중 상태는 핵심 공식과 직접 관계가 없다고 판단
drop_cols = ["Body_Temperature(F)", "Weight_Status", "Height(Feet)", "Height(Remainder_Inches)"]

train_x2 = train_x.drop(columns=drop_cols)
test_x2  = test_x.drop(columns=drop_cols)

poly2 = PolynomialFeatures(degree=3, interaction_only=True, include_bias=False)
ridge2 = Ridge(alpha=0.004)
lr2    = LinearRegression()

X_train2, X_val2, y_train2, y_val2 = train_test_split(
    train_x2, train_y, test_size=0.2, random_state=777
)

X_train2_poly = poly2.fit_transform(X_train2)
X_val2_poly   = poly2.transform(X_val2)

lr2.fit(X_train2_poly, y_train2)
ridge2.fit(X_train2_poly, y_train2)

val_pred2 = 0.45 * np.round(lr2.predict(X_val2_poly)) + 0.55 * np.round(ridge2.predict(X_val2_poly))
rmse2 = np.sqrt(mean_squared_error(y_val2, val_pred2))
print(f"2차 시도 Validation RMSE: {rmse2:.5f}")

---
## 3차 시도 — Keytel 공식 역공학 + CatBoost 잔차 학습 ✅ 최종 제출

### 핵심 전환점
RMSE가 일정 수준에서 계속 막히는 이유를 분석하다가,  
트리 모델이 **0.5 단위 반올림 경계선**을 맞추려고 리프 노드를 과도하게 뻗어  
전체 규칙을 망가뜨리고 있다는 걸 발견했다.

심박수와 칼로리 소모량 사이의 선형 관계를 의심하고 생리학 논문을 탐색한 결과,  
데이터가 **Keytel 공식** 기반으로 생성됐다는 것을 찾아냈다.

> Keytel LR et al. (2005). Prediction of energy expenditure from heart rate monitoring  
> during submaximal exercise. *J Sports Sci*, 23(3), 289–297.

### 최종 파이프라인
```
[입력 데이터]
     ↓
[Stage 1] Keytel 공식 기반 Ridge  →  기본 예측값 (선형)
     ↓
[Stage 2] CatBoost               →  잔차(오차) 예측 및 보정
     ↓
[최종 예측값 반올림]
     ↓
검증 RMSE 0.0529
```

### 피처 엔지니어링 설계 의도

| 변수명 | 생리학적 근거 |
|--------|-------------|
| `Metabolic_Factor` | 체온 1°C 상승 시 대사율 약 13% 증가 (Q10 효과) |
| `EPOC_Factor` | 고온 + 고강도 운동 시 운동 후 초과산소소모량 증가 |
| `Thermal_Strain` | 체온 × 운동시간 복합 열부하 누적 효과 |
| `HRR_Ratio` | Karvonen 공식 기반 심박수 예비력 대비 강도 비율 |

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("catboost")
install("optuna")

from catboost import CatBoostRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

SEED = 42
np.random.seed(SEED)
print("라이브러리 로드 완료")

In [ ]:
DATA_DIR = "/content/"
train = pd.read_csv(DATA_DIR + "train.csv")
test  = pd.read_csv(DATA_DIR + "test.csv")

TARGET    = "Calories_Burned"
train_x   = train.drop(columns=["ID", TARGET]).copy()
train_y   = train[TARGET].copy()
test_x    = test.drop(columns=["ID"], errors="ignore").copy()

print(f"Train: {train_x.shape} | Test: {test_x.shape}")

In [ ]:
# Weight_Status: Ordinal Encoding (순서 의미 반영)
status_map = {"Normal Weight": 0, "Overweight": 1, "Obese": 2}
for df in [train_x, test_x]:
    df["Weight_Status"] = df["Weight_Status"].map(status_map)

# Gender: Label Encoding — Keytel 공식에서 남/여 분기에 직접 사용
le = LabelEncoder()
le.fit(train_x["Gender"])
train_x["Gender"] = le.transform(train_x["Gender"])
test_x["Gender"]  = le.transform(test_x["Gender"])

MALE_CODE = int(le.transform(["M"])[0])
print(f"Gender 인코딩 — 남성(M) 코드: {MALE_CODE}")

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    생리학 기반 파생 변수 생성.

    단순 수치 조합이 아니라 각 변수의 생리학적 의미를 반영해 설계했다.
    - Metabolic_Factor : 체온에 따른 대사 속도 가속화 (Q10 효과)
    - EPOC_Factor      : 고온 + 고강도 운동 시 회복 산소소모량 증가
    - Thermal_Strain   : 누적 열부하 (체온 × 운동시간)
    - HRR_Ratio        : 심박수 예비력 기반 운동 강도 (Karvonen 공식)
    """
    df = df.copy()

    # 단위 변환
    df["Height_inches"] = df["Height(Feet)"] * 12 + df["Height(Remainder_Inches)"]
    df["Height_m"]      = df["Height_inches"] * 0.0254
    df["Weight_kg"]     = df["Weight(lb)"] / 2.2046
    df["Temp_C"]        = (df["Body_Temperature(F)"] - 32) * 5 / 9

    # 신체 구성 지표
    df["BMI"]     = df["Weight_kg"] / (df["Height_m"] ** 2)
    df["BMI_Age"] = df["BMI"] * df["Age"]

    # 운동 강도 지표
    df["Intensity"]        = df["BPM"] * df["Exercise_Duration"]
    df["Intensity_Weight"] = df["Intensity"] * df["Weight_kg"]

    # 체온 관련 생리학 지표
    df["Metabolic_Factor"] = 1.13 ** (df["Temp_C"] - 37)          # Q10 효과
    df["EPOC_Factor"]      = df["Intensity"] * (df["Temp_C"] - 36.5)  # 고온 + 고강도 시너지
    df["Thermal_Strain"]   = df["Temp_C"] * df["Exercise_Duration"]    # 누적 열부하
    df["Heat_Stress_Index"] = df["Temp_C"] * df["BPM"]

    # 심박수 기반 강도 지표 (Karvonen)
    df["Max_HR"]    = 220 - df["Age"]
    df["HRR_Ratio"] = (df["BPM"] - 60) / (df["Max_HR"] - 60)

    # 나이 관련
    df["Age_BPM"]     = df["Age"] * df["BPM"]
    df["Age_squared"] = df["Age"] ** 2

    # BMR (Mifflin-St Jeor)
    df["BMR"]          = (10 * df["Weight_kg"]) + (6.25 * df["Height_m"] * 100) - (5 * df["Age"])
    df["Gender_BMR"]   = df["Gender"] * df["BMR"]
    df["Gender_Weight"]= df["Gender"] * df["Weight_kg"]
    df["Work_Efficiency"] = df["Exercise_Duration"] / (df["BPM"] + 1)

    return df


train_x = add_features(train_x)
test_x  = add_features(test_x)

print(f"피처 엔지니어링 완료 — 최종 피처 수: {train_x.shape[1]}")

In [ ]:
# Keytel 공식 파라미터
# 논문 원본 계수를 기준으로 데이터 역공학을 통해 검증한 값
best_params = [
    2.01715651e-01,   # 남성: age 계수
    4.09789570e-02,   # 남성: weight(lb) 계수
    6.30933782e-01,   # 남성: BPM 계수
   -5.50993643e+01,   # 남성: 상수
    7.40152512e-02,   # 여성: age 계수
   -5.74001235e-02,   # 여성: weight(kg) 계수
    4.47193226e-01,   # 여성: BPM 계수
   -2.04026456e+01,   # 여성: 상수
]


def predict_keytel(df: pd.DataFrame, params: list) -> np.ndarray:
    """
    Keytel 공식 기반 칼로리 raw 예측.

    남성과 여성의 공식이 다르며, Gender 인코딩 결과(MALE_CODE)로 분기한다.
    - 남성: Weight를 lb 단위로 직접 사용
    - 여성: Weight를 kg으로 변환 후 사용 (논문 원본 공식 반영)
    """
    df = df.copy()
    (m_age, m_wlb, m_bpm, m_const,
     f_age, f_wkg, f_bpm, f_const) = params

    weight_kg = df["Weight(lb)"] / 2.2046

    cal_m = (
        df["Age"] * m_age + df["Weight(lb)"] * m_wlb + df["BPM"] * m_bpm + m_const
    ) * df["Exercise_Duration"] / 4.184

    cal_f = (
        df["Age"] * f_age + weight_kg * f_wkg + df["BPM"] * f_bpm + f_const
    ) * df["Exercise_Duration"] / 4.184

    return np.where(df["Gender"] == MALE_CODE, cal_m, cal_f)


def finalize_pred(raw_pred: np.ndarray) -> np.ndarray:
    return np.clip(np.floor(raw_pred + 0.5), 0, None)


print("Keytel 공식 정의 완료")

In [ ]:
# 체온 40°C 이상 구간에서 반올림 경계 오차가 발생하는 이상치 조건 기반 처리
# 하드코딩 제거 대신 조건 기반으로 처리해 재사용성 확보
villain_mask = train["Body_Temperature(F)"] >= 104
villain_idx  = train[villain_mask].index.tolist()
print(f"Rounding Paradox 해당 샘플 수: {len(villain_idx)}건")

base_x = train_x.drop(index=villain_idx, errors="ignore").copy()
base_y = train_y.drop(index=villain_idx, errors="ignore").copy()

X_train, X_val, y_train, y_val = train_test_split(
    base_x, base_y, test_size=0.2, random_state=SEED
)

# Keytel 기본 예측 및 잔차 계산
train_raw      = predict_keytel(X_train, best_params)
train_pred     = finalize_pred(train_raw)
train_residual = y_train - train_pred

keytel_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
print(f"Keytel 공식 Train RMSE: {keytel_rmse:.4f}")

In [ ]:
# Optuna로 CatBoost 하이퍼파라미터 탐색
# CATBOOST_TUNE = False로 바꾸면 기본값 사용
CATBOOST_TUNE = True
OPTUNA_TRIALS = 100

if CATBOOST_TUNE:
    print(f"CatBoost 하이퍼파라미터 탐색 시작 ({OPTUNA_TRIALS} trials)...")

    def cb_objective(trial):
        params = {
            "iterations":       trial.suggest_int("iterations", 200, 1000),
            "depth":            trial.suggest_int("depth", 4, 8),
            "learning_rate":    trial.suggest_float("learning_rate", 0.01, 0.15),
            "l2_leaf_reg":      trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 1, 50),
            "random_state": SEED,
            "verbose": 0,
        }
        model = CatBoostRegressor(**params)
        model.fit(X_train, train_residual)

        val_raw_      = predict_keytel(X_val, best_params)
        val_residual_ = model.predict(X_val)
        val_final_    = finalize_pred(val_raw_ + val_residual_)

        return np.sqrt(mean_squared_error(y_val, val_final_))

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(seed=SEED)
    )
    study.optimize(cb_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

    best_cb_params = study.best_params
    best_cb_params["random_state"] = SEED
    best_cb_params["verbose"]      = 0
    print(f"최적 Val RMSE: {study.best_value:.6f}")

else:
    best_cb_params = {
        "iterations": 500, "depth": 5, "learning_rate": 0.05,
        "random_state": SEED, "verbose": 0,
    }

res_model = CatBoostRegressor(**best_cb_params)
res_model.fit(X_train, train_residual)
print("CatBoost 잔차 모델 학습 완료")

In [ ]:
# Validation 성능 확인
val_raw      = predict_keytel(X_val, best_params)
val_residual = res_model.predict(X_val)
val_final    = finalize_pred(val_raw + val_residual)

rmse = np.sqrt(mean_squared_error(y_val, val_final))
print(f"최종 Validation RMSE: {rmse:.6f}")

# Test 예측
test_raw      = predict_keytel(test_x, best_params)
test_residual = res_model.predict(test_x)
test_final    = finalize_pred(test_raw + test_residual)

In [ ]:
submission = pd.read_csv(DATA_DIR + "sample_submission.csv")
submission[TARGET] = test_final
submission.to_csv("submission.csv", index=False)
print("submission.csv 저장 완료")
submission.head()

---
## 미제출 시도 — ElasticNet 기반 잔차 학습

마감 시간에 맞추지 못했지만, 잔차 보정 모델로 CatBoost 대신  
L1 + L2 규제를 함께 쓰는 ElasticNetCV를 시도해봤다.

상호작용 피처(운동시간×나이, BPM×체중 등)를 추가하고  
StandardScaler로 스케일링한 후 잔차를 학습시켰다.  
시간이 더 있었다면 CatBoost와 앙상블해볼 여지가 있었다.

In [ ]:
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler

def add_interaction_features(df):
    df = df.copy()
    df['E_A_I'] = df['Exercise_Duration'] * df['Age']    # 운동시간 × 나이
    df['B_W_I'] = df['BPM'] * df['Weight(lb)']           # BPM × 체중
    df['B_A_I'] = df['BPM'] * df['Age']                  # BPM × 나이
    return df


def apply_keytel_simple(df):
    w_kg = df['Weight(lb)'] / 2.2046
    cal_male   = ((df['Age'] * 0.2017) + (w_kg * 0.09036)  + (df['BPM'] * 0.6309) - 55.0969) * df['Exercise_Duration'] / 4.184
    cal_female = ((df['Age'] * 0.074)  - (w_kg * 0.05741)  + (df['BPM'] * 0.4472) - 20.4022) * df['Exercise_Duration'] / 4.184
    return np.where(df['Gender'] == 1, cal_male, cal_female)


train_x_e = add_interaction_features(train_x)
test_x_e  = add_interaction_features(test_x)

X_tr, X_vl, y_tr, y_vl = train_test_split(train_x_e, train_y, test_size=0.2, random_state=42)

X_tr = X_tr.copy()
X_vl = X_vl.copy()
X_tr['Keytel_Pred'] = apply_keytel_simple(X_tr)
X_vl['Keytel_Pred'] = apply_keytel_simple(X_vl)

features_res = ['Keytel_Pred', 'E_A_I', 'B_W_I', 'B_A_I']
y_residual   = y_tr - X_tr['Keytel_Pred']

scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_tr[features_res])
X_vl_scaled = scaler.transform(X_vl[features_res])

elastic_model = ElasticNetCV(cv=5, random_state=42, n_jobs=-1)
elastic_model.fit(X_tr_scaled, y_residual)

val_pred_e = X_vl['Keytel_Pred'] + elastic_model.predict(X_vl_scaled)
val_pred_e = np.round(val_pred_e.clip(lower=0))

rmse_e = np.sqrt(mean_squared_error(y_vl, val_pred_e))
print(f"ElasticNet 잔차 모델 RMSE: {rmse_e:.5f}")
print(f"최적 alpha: {elastic_model.alpha_:.5f} | l1_ratio: {elastic_model.l1_ratio_:.5f}")